In [0]:
# Databricks notebook source
# Simulates a source-system extract landing in a Volume, then loads it raw into Bronze.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "churn_dev")
catalog, schema = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.landing")

In [0]:
# COMMAND ----------
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 20000
df = pd.DataFrame({
    "customer_id": np.arange(n),
    "tenure_months": rng.integers(1, 72, n),
    "contract": rng.choice(["month-to-month", "one-year", "two-year"], n, p=[.55, .25, .20]),
    "monthly_charges": rng.normal(70, 25, n).clip(20, 150).round(2),
    "support_calls": rng.poisson(1.5, n),
    "internet_service": rng.choice(["fiber", "dsl", "none"], n, p=[.45, .40, .15]),
})
logit = (-1.0 - 0.04 * df.tenure_months + 0.02 * (df.monthly_charges - 70)
         + 0.35 * df.support_calls
         + np.where(df.contract == "month-to-month", 1.0,
                    np.where(df.contract == "one-year", -0.2, -1.0)))
df["churned"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

# Realistic dirt: missing values and duplicate rows
df.loc[rng.choice(n, 300, replace=False), "monthly_charges"] = np.nan
df = pd.concat([df, df.sample(200, random_state=1)])

path = f"/Volumes/{catalog}/{schema}/landing/customers.csv"
df.to_csv(path, index=False)

In [0]:
# COMMAND ----------
from pyspark.sql import functions as F

raw = (spark.read.option("header", True).option("inferSchema", True).csv(path)
       .withColumn("_ingested_at", F.current_timestamp())
       .withColumn("_source_file", F.lit(path)))
raw.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.bronze_customers")
print(f"Bronze rows: {raw.count()}")